In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

import qwen_vl

In [2]:
graph_root = Path("../output/cholecseg8k/video01_00080_qwen_cat/graph")

In [3]:
feature_files = sorted((graph_root / "c_qwen_feats").glob("*.npy"), key=lambda f: int(f.stem))
qwen_features = [np.load(f) for f in feature_files]
print("n_features", [f.shape[0] for f in qwen_features])
adjacency_matrices = np.load(graph_root / "graph.npy")
print("adjacency_matrices", adjacency_matrices.shape, (adjacency_matrices > 0.0).sum() / adjacency_matrices.size)
centers = np.load(graph_root / "c_centers.npy")
print("centers", centers.shape)
centroids = np.load(graph_root / "c_centroids.npy")
print("centroids", centroids.shape)
extents = np.load(graph_root / "c_extents.npy")
print("extents", extents.shape)

n_features [32, 35, 34, 34, 36, 34, 36, 36, 10, 32, 39, 66, 36, 34, 35, 36, 12, 38, 35, 14, 42, 4, 12, 27, 32, 32, 38, 66, 20]
adjacency_matrices (5, 12, 12) 0.18611111111111112
centers (5, 12, 3)
centroids (5, 12, 3)
extents (5, 12, 3)


In [4]:
adjacency_matrices = adjacency_matrices[:1]
centers = centers[:1]
centroids = centroids[:1]
extents = extents[:1]

In [5]:
model, processor = qwen_vl.get_patched_qwen()

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [21]:
# p = "../data/cholecseg8k/preprocessed_ssg/video43/video43_00147/qwen_instance_features/frame_000060_f.npy"
# p = "/home/tumai/nico/surgery-scene-graphs/data/cholecseg8k/preprocessed_ssg/video43/video43_00147/qwen_instance_features/000060_f.npy"
p = "/home/tumai/nico/surgery-scene-graphs/data/cholecseg8k/preprocessed_ssg/video01/video01_00080/qwen_instance_features/000078_f.npy"
f = np.load(p)
qwen_vl.ask_qwen_about_image_features(torch.tensor(f), "Is there a surgical tool visible? If so, what is it touching?", model, processor, system_prompt="You are an expert visceral surgeon.")

'Yes, there is a surgical tool visible in the image. The tool appears to be a scalpel or a similar sharp instrument used for making precise cuts during surgery. It is touching the tissue, likely as part of a surgical procedure.'

In [33]:
# systemprompt = """
# You are an assistant that answers questions about scene graphs
# of a cholecystectomy procedure.

# A scene graph represents a 3D scene through time as
# a list of graphs, where each graphs corresponds to a
# timestep.
# Each graph contains a set of nodes and edges.
# Each node corresponds to an object present in the scene,
# and associated properties like its position, extent,
# etc. at the respective timestep.
# Edge weights correspond to distances between objects
# at the respective timestep.
# Each object is represented by an image of the
# object.
# The scene graph as a whole is given as a list of
# objects followed by a list of graphs.

# You will be given a scene graph and a question by the user.
# You answer the question ONLY based on the scene graph
# and context provided by this system prompt.

# You answer with nothing but the response to the question.
# You keep the response as concise as possible, while
# answering all aspects the question.
# """

systemprompt = """
You are an expert visceral surgeon that answers questions about a cholecystectomy procedure.

A scene graph represents a 3D scene through time as
a list of graphs, where each graphs corresponds to a
timestep.
Each graph contains a set of nodes and edges.
Each node corresponds to an object present in the scene,
and associated properties like its position, extent,
etc. at the respective timestep. Please interpret those in latent space.
Edge weights correspond to distances between objects
at the respective timestep.
The scene graph as a whole is given as a list of
objects followed by a list of graphs.
You should interpret the nodes in latent space, then look at the graph structure and edge weights over time. Reason about what is happening across the scene graph.
Precisely refer to the nodes and timesteps and interpret them in latent space.
You may be uncertain but be assured that this is a surgical scene in cholecystectomy.
You should reason clearly about what are tools and which anatomy and confidently state your assessments.
Do not beat around the bush, avoid terms like "likely" or "possibly". Talk like an expert surgeon.
"""

In [39]:
qwen_vl.prompt_with_graph(
    question="What anatomy are you seeing? And which nodes correspond to what?",
    node_feats=qwen_features,
    adjacency_matrices=adjacency_matrices,
    node_centers=centers,
    node_centroids=centroids,
    node_extents=extents,
    model=model,
    processor=processor,
)

"The scene graph appears to depict a medical imaging view, likely a laparoscopic or endoscopic examination of internal organs. \n\n- Node `0` likely corresponds to the patient's abdomen or part of the abdominal cavity.\n- Node `1` could represent a specific organ or structure within the abdomen, possibly the liver given the coloration and texture.\n- Node `2` might be another organ or structure, potentially the spleen or another part of the gastrointestinal tract.\n- Node `3` seems to be a different organ or structure, possibly the stomach or another part of the gastrointestinal tract.\n- Node `4` could be a structure"